In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
from pathlib import Path

In [19]:
def extract_params(filename):
    """Extrae lr, beta1, beta2 del nombre del archivo"""
    match = re.search(r'lr_([\d.]+?)_beta1_([\d.]+?)_beta2_([\d.]+?)(?:\.csv|$)', filename)
    if match:
        lr_str = match.group(1).rstrip('.')
        beta1_str = match.group(2).rstrip('.')
        beta2_str = match.group(3).rstrip('.')
        
        return {
            'lr': float(lr_str),
            'beta1': float(beta1_str),
            'beta2': float(beta2_str)
        }
    return None

# Function to determine the type (continuous or discrete)
def get_type(filename):
    """Determines if it is continuous or discrete"""
    if 'continuous' in filename:
        return 'continuous'
    elif 'discrete' in filename:
        return 'discrete'
    return None

# Read all CSV files
results_dir = Path("results")
csv_files = list(results_dir.glob("*.csv"))

# Organize files by parameters
data_dict = {}
for csv_file in csv_files:
    params = extract_params(str(csv_file))
    if params is None:
        continue
    
    file_type = get_type(str(csv_file))
    if file_type is None:
        continue
    
    key = (params['lr'], params['beta1'], params['beta2'])
    if key not in data_dict:
        data_dict[key] = {}
    
    df = pd.read_csv(csv_file)
    data_dict[key][file_type] = df

In [20]:
c = 1.0

In [21]:
for (lr, beta1, beta2), data in data_dict.items():
    # Check that we have both types
    if 'continuous' not in data or 'discrete' not in data:
        print(f"Warning: Missing data for lr={lr}, beta1={beta1}, beta2={beta2}")
        continue
    
    df_cont = data['continuous']
    df_disc = data['discrete']
    
    # Extract data
    t_cont = df_cont["t"].to_numpy()
    theta_cont = df_cont["theta_history"].to_numpy()
    m_cont = df_cont["m_history"].to_numpy()
    v_cont = df_cont["v_history"].to_numpy()
    
    t_disc = df_disc["t"].to_numpy()
    theta_disc = df_disc["theta_history"].to_numpy()
    m_disc = df_disc["m_history"].to_numpy()
    v_disc = df_disc["v_history"].to_numpy()
    
    # Calculate f(theta) for both
    f_cont = (1 - theta_cont)**2 + c * (theta_cont**2 - 1)**2
    f_disc = (1 - theta_disc)**2 + c * (theta_disc**2 - 1)**2
    
    # ========== FIG 1: f(theta_k) vs iteration ==========
    plt.figure(figsize=(10, 6))
    plt.semilogy(t_cont, f_cont, label='Continuous Adam', linewidth=2)
    plt.semilogy(t_disc, f_disc, label='Discrete Adam', linewidth=2, linestyle='--')
    plt.xlabel("iteration t")
    plt.ylabel("f(theta_t)")
    plt.title(f"Convergence: f(theta_t) vs t\n(lr={lr}, β₁={beta1}, β₂={beta2})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"results/convergence_lr_{lr}_beta1_{beta1}_beta2_{beta2}.png", dpi=150)
    plt.close()
    
    # ========== FIG 2: paisaje f(theta) + trayectoria ==========
    th_min = min(theta_cont.min(), theta_disc.min()) - 1.0
    th_max = max(theta_cont.max(), theta_disc.max()) + 1.0
    th_grid = np.linspace(th_min, th_max, 2000)
    f_grid = (1 - th_grid)**2 + c * (th_grid**2 - 1)**2
    
    # Subsampling for visualization
    n_show = 300
    idx_cont = np.linspace(0, len(theta_cont) - 1, n_show).astype(int)
    idx_disc = np.linspace(0, len(theta_disc) - 1, n_show).astype(int)
    
    plt.figure(figsize=(10, 6))
    plt.plot(th_grid, f_grid, 'k-', linewidth=1.5, alpha=0.5, label='Paisaje f(θ)')
    
    # Continuous trajectory
    plt.plot(theta_cont[idx_cont], f_cont[idx_cont], '-', linewidth=2, 
             label='Continuous Adam', alpha=0.7)
    plt.scatter(theta_cont[idx_cont], f_cont[idx_cont], s=20, alpha=0.6)
    
    # Discrete trajectory
    plt.plot(theta_disc[idx_disc], f_disc[idx_disc], '--', linewidth=2, 
             label='Discrete Adam', alpha=0.7)
    plt.scatter(theta_disc[idx_disc], f_disc[idx_disc], s=20, alpha=0.6)
    
    # Highlight start and end
    plt.scatter([theta_cont[0]], [f_cont[0]], s=100, marker="o", 
                color='blue', label='Start Continuous', zorder=5)
    plt.scatter([theta_cont[-1]], [f_cont[-1]], s=100, marker="X", 
                color='blue', label='Final Continuous', zorder=5)
    plt.scatter([theta_disc[0]], [f_disc[0]], s=100, marker="o", 
                color='orange', label='Start Discrete', zorder=5)
    plt.scatter([theta_disc[-1]], [f_disc[-1]], s=100, marker="X", 
                color='orange', label='Final Discrete', zorder=5)
    
    plt.xlabel("theta")
    plt.ylabel("f(theta)")
    plt.title(f"Landscape 1D + trajectory of iterations\n(lr={lr}, β₁={beta1}, β₂={beta2})")
    plt.legend(loc='best', fontsize=8)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"results/landscape_lr_{lr}_beta1_{beta1}_beta2_{beta2}.png", dpi=150)
    plt.close()
    
    # ========== FIG 3: theta vs iteración ==========
    plt.figure(figsize=(10, 6))
    plt.plot(t_cont, theta_cont, label='Continuous Adam', linewidth=2)
    plt.plot(t_disc, theta_disc, label='Discrete Adam', linewidth=2, linestyle='--')
    plt.xlabel("iteration t")
    plt.ylabel("theta_t")
    plt.title(f"Evolution of theta_t\n(lr={lr}, β₁={beta1}, β₂={beta2})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"results/theta_evolution_lr_{lr}_beta1_{beta1}_beta2_{beta2}.png", dpi=150)
    plt.close()
    
    # ========== FIG 4: m_history vs iteración ==========
    plt.figure(figsize=(10, 6))
    plt.plot(t_cont, m_cont, label='Continuous Adam', linewidth=2)
    plt.plot(t_disc, m_disc, label='Discrete Adam', linewidth=2, linestyle='--')
    plt.xlabel("iteration t")
    plt.ylabel("m_t")
    plt.title(f"Evolution of the moment m_t\n(lr={lr}, β₁={beta1}, β₂={beta2})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"results/momentum_lr_{lr}_beta1_{beta1}_beta2_{beta2}.png", dpi=150)
    plt.close()
    
    # ========== FIG 5: v_history vs iteración ==========
    plt.figure(figsize=(10, 6))
    plt.plot(t_cont, v_cont, label='Continuous Adam', linewidth=2)
    plt.plot(t_disc, v_disc, label='Discrete Adam', linewidth=2, linestyle='--')
    plt.xlabel("iteration t")
    plt.ylabel("v_t")
    plt.title(f"Evolution of the adaptive variance v_t\n(lr={lr}, β₁={beta1}, β₂={beta2})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"results/variance_lr_{lr}_beta1_{beta1}_beta2_{beta2}.png", dpi=150)
    plt.close()
    
    print(f"Plots generated for lr={lr}, beta1={beta1}, beta2={beta2}")

Plots generated for lr=0.01, beta1=0.9, beta2=0.999
Plots generated for lr=0.001, beta1=0.9, beta2=0.999
